# Notebook 6: Interactive Chat Interface

## Purpose
This notebook provides a fully interactive chat session with the Smart Pantry Chef.
Run all setup cells, then use the chat cell at the bottom to have a live conversation.

Each message shows which tools the agent called, making the agentic reasoning transparent.

> **Prerequisite:** Run `01_database_setup.ipynb` first to initialize the pantry.

---

## 6.1 Full setup (self-contained)

In [ ]:
import mysql.connector, json, os, time
from datetime import date, datetime
from pathlib import Path
import sql_openai_config
from openai import OpenAI

PROJECT_ROOT = Path(os.getcwd()).parent

MYSQL_CONFIG = sql_openai_config.get_mysql_config()

os.environ["OPENAI_API_KEY"] = sql_openai_config.get_openai()

client = OpenAI()

def get_connection():
    return mysql.connector.connect(**MYSQL_CONFIG)


def get_pantry_items():
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT name, category, quantity, unit, expiry_date "
        "FROM pantry ORDER BY expiry_date"
    )
    rows = cursor.fetchall()
    conn.close()

    today = date.today()
    return [
        {
            "name": n,
            "category": c,
            "quantity": q,
            "unit": u,
            "expiry_date": str(e),
            "days_until_expiry": (
                (datetime.strptime(str(e), "%Y-%m-%d").date() - today).days
                if e
                else None
            ),
        }
        for n, c, q, u, e in rows
    ]

def get_at_risk_items(threshold_days=3):
    return [
        {
            **item,
            "status": "EXPIRED" if item["days_until_expiry"] < 0 else "AT RISK",
        }
        for item in get_pantry_items()
        if item["days_until_expiry"] is not None
        and item["days_until_expiry"] <= threshold_days
    ]


def add_pantry_item(name, category, quantity, unit, expiry_date):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "INSERT INTO pantry (name, category, quantity, unit, expiry_date) "
        "VALUES (%s, %s, %s, %s, %s)",
        (name, category, quantity, unit, expiry_date),
    )
    conn.commit()
    conn.close()
    return f"Added '{name}' (expires {expiry_date})."


def remove_pantry_item(name):
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "DELETE FROM pantry WHERE LOWER(name) = LOWER(%s)",
        (name,),
    )
    conn.commit()
    affected = cursor.rowcount
    conn.close()
    return f"Removed '{name}'." if affected > 0 else f"'{name}' not found."


def update_quantity(name, new_quantity):
    if new_quantity <= 0:
        return remove_pantry_item(name)
    conn = get_connection()
    cursor = conn.cursor()
    cursor.execute(
        "UPDATE pantry SET quantity = %s WHERE LOWER(name) = LOWER(%s)",
        (new_quantity, name),
    )
    conn.commit()
    affected = cursor.rowcount
    conn.close()
    return (
        f"Updated '{name}' to {new_quantity}."
        if affected > 0
        else f"'{name}' not found."
    )


TOOL_DEFINITIONS = [
    {'type':'function','function':{'name':'get_pantry_items','description':'Get all pantry items with expiry info. Always call before suggesting recipes.','parameters':{'type':'object','properties':{},'required':[]}}},
    {'type':'function','function':{'name':'get_at_risk_items','description':'Get items expiring within threshold_days days (default 3). Use for expiry prioritization.','parameters':{'type':'object','properties':{'threshold_days':{'type':'integer','description':'Days threshold'}},'required':[]}}},
    {'type':'function','function':{'name':'add_pantry_item','description':'Add a new pantry ingredient.','parameters':{'type':'object','properties':{'name':{'type':'string'},'category':{'type':'string'},'quantity':{'type':'number'},'unit':{'type':'string'},'expiry_date':{'type':'string','description':'YYYY-MM-DD'}},'required':['name','category','quantity','unit','expiry_date']}}},
    {'type':'function','function':{'name':'remove_pantry_item','description':'Remove a fully used or discarded ingredient.','parameters':{'type':'object','properties':{'name':{'type':'string'}},'required':['name']}}},
    {'type':'function','function':{'name':'update_quantity','description':'Update remaining quantity of an ingredient after partial use.','parameters':{'type':'object','properties':{'name':{'type':'string'},'new_quantity':{'type':'number'}},'required':['name','new_quantity']}}}
]

TOOL_MAP = {
    'get_pantry_items'   : lambda a: get_pantry_items(),
    'get_at_risk_items'  : lambda a: get_at_risk_items(**a),
    'add_pantry_item'    : lambda a: add_pantry_item(**a),
    'remove_pantry_item' : lambda a: remove_pantry_item(**a),
    'update_quantity'    : lambda a: update_quantity(**a),
}

SYSTEM_PROMPT = """
You are the Smart Pantry Chef, an AI kitchen assistant specialized in zero-waste cooking.
Your primary goal is to help users reduce household food waste by intelligently using
ingredients before they expire.

## Your tools
- get_pantry_items()    → full inventory with expiry dates
- get_at_risk_items()   → items expiring within N days
- add_pantry_item()     → add newly purchased ingredients
- remove_pantry_item()  → remove used/discarded items
- update_quantity()     → update remaining quantity

## Reasoning rules
1. NEVER suggest an ingredient not confirmed in the pantry. Always call tools first.
2. ALWAYS prioritize at-risk ingredients (expiring within 3 days).
3. Only list pantry-confirmed ingredients. Salt and water may be labeled (assumed).
4. After every recipe, offer to update the pantry inventory.

## Recipe output format
**Recipe:** [Name]
**Why this recipe?** [At-risk ingredients, days remaining]
**Ingredients from your pantry:** [list]
**Assumed staples:** (if any)
**Instructions:** 1-8 steps
**Pantry update:** Offer to update.

## Tone: warm, encouraging, practical.
"""

print('Smart Pantry Chef ready!')
print('Use the chat cell below to start your conversation.')

Smart Pantry Chef ready!
Use the chat cell below to start your conversation.


## 6.2 Pantry snapshot — see what's in your fridge right now

In [20]:
items   = get_pantry_items()
at_risk = get_at_risk_items(threshold_days=7)

print(f'Total pantry items: {len(items)}')
print(f'\n🚨 AT RISK (within 7 days): {len(at_risk)}')
for item in at_risk:
    flag = '❌ EXPIRED' if item['days_until_expiry'] < 0 else f"⚠️  {item['days_until_expiry']} days left"
    print(f"  {item['name']:<20} {flag}")

print(f'\n✅ OK (8+ days):  {len(items) - len(at_risk)} items')

Total pantry items: 176

🚨 AT RISK (within 7 days): 30
  eggs                 ❌ EXPIRED
  Ground turkey        ❌ EXPIRED
  Avocado              ❌ EXPIRED
  tomato               ⚠️  2 days left
  banana               ⚠️  2 days left
  bread whole wheat    ⚠️  2 days left
  croissants           ⚠️  2 days left
  milk                 ⚠️  3 days left
  salmon fillet        ⚠️  3 days left
  lettuce romaine      ⚠️  3 days left
  mushrooms button     ⚠️  3 days left
  raspberries          ⚠️  3 days left
  bread sourdough      ⚠️  3 days left
  chicken breast       ⚠️  4 days left
  kale                 ⚠️  4 days left
  mushrooms cremini    ⚠️  4 days left
  strawberries         ⚠️  4 days left
  burger buns          ⚠️  4 days left
  bagels               ⚠️  4 days left
  orange juice         ⚠️  4 days left
  yogurt               ⚠️  5 days left
  broccoli             ⚠️  5 days left
  blueberries          ⚠️  5 days left
  naan                 ⚠️  5 days left
  tilapia fillets      ⚠️  

## 6.3 Conversation session

Run this cell to start a fresh conversation. The `conversation` list maintains memory across turns.

Suggested opening messages to try:
- *"What should I cook tonight to use up what's expiring?"*
- *"Can you suggest a breakfast using items that expire soon?"*
- *"I just bought 300g of ground turkey expiring 2025-03-28. Add it."*
- *"What's in my pantry right now?"*
- *"I made the recipe — I used all the spinach and 1 piece of chicken."*

In [21]:
# Initialize a fresh conversation (run this once per session)
conversation = []
print('New conversation started. Run the chat cell below to send a message.')

New conversation started. Run the chat cell below to send a message.


In [22]:
# ── CHAT CELL — edit USER_MESSAGE and run this cell to chat ──────────────────

USER_MESSAGE = "What should I cook tonight to use up what's about to expire?"

#USER_MESSAGE = input("You: ")

if USER_MESSAGE.strip().lower() in {"exit", "quit"}:
    print("Ending chat.")
else:
    print(f"\nYou: {USER_MESSAGE}\n")
    conversation.append({"role": "user", "content": USER_MESSAGE})

    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + conversation
    tool_call_count = 0
    tools_this_turn = []

    while True:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=TOOL_DEFINITIONS,
            tool_choice="auto",
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            reply = msg.content
            break

        messages.append(msg)
        for tc in msg.tool_calls:
            fn_name = tc.function.name
            fn_args = json.loads(tc.function.arguments)
            tool_call_count += 1
            tools_this_turn.append(f"{fn_name}({fn_args})")
            result = TOOL_MAP[fn_name](fn_args) if fn_name in TOOL_MAP else "ERROR"
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, default=str),
                }
            )

    conversation.append({"role": "assistant", "content": reply})
    print(f"\nChef:\n{reply}\n")
    print(f"({tool_call_count} tool call(s): {tools_this_turn})")

# ─────────────────────────────────────────────────────────────────────────────
# You do not need to edit anything below this line

# print(f'You: {USER_MESSAGE}\n')
# conversation.append({'role': 'user', 'content': USER_MESSAGE})

# messages         = [{'role': 'system', 'content': SYSTEM_PROMPT}] + conversation
# tool_call_count  = 0
# tools_this_turn  = []

# while True:
#     response = client.chat.completions.create(
#         model='gpt-4o-mini', messages=messages,
#         tools=TOOL_DEFINITIONS, tool_choice='auto'
#     )
#     msg = response.choices[0].message

#     if not msg.tool_calls:
#         reply = msg.content
#         break

#     messages.append(msg)
#     for tc in msg.tool_calls:
#         fn_name = tc.function.name
#         fn_args = json.loads(tc.function.arguments)
#         tool_call_count += 1
#         tools_this_turn.append(fn_name)
#         print(f'  [tool {tool_call_count}: {fn_name}({fn_args})]')
#         result = TOOL_MAP[fn_name](fn_args) if fn_name in TOOL_MAP else 'ERROR'
#         messages.append({'role':'tool','tool_call_id':tc.id,'content':json.dumps(result,default=str)})

# conversation.append({'role': 'assistant', 'content': reply})

# print(f'\n[{tool_call_count} tool call(s): {tools_this_turn}]')
# print(f'\nChef:\n{"-"*50}\n{reply}\n{"-"*50}')
# print(f'\n[Conversation turn {len([m for m in conversation if m["role"]=="user"])} complete]')


You: What should I cook tonight to use up what's about to expire?


Chef:
Here’s a great recipe to use up some of your at-risk ingredients!

**Recipe:** Salmon and Mushroom Toast  
**Why this recipe?** Uses up salmon fillet (3 days), mushrooms (3 days), and whole wheat bread (2 days).  
**Ingredients from your pantry:**  
- salmon fillet (400 grams)  
- mushrooms button (400 grams)  
- bread whole wheat (1 loaf)  

**Assumed staples:** olive oil, salt, pepper

**Instructions:**  
1. Preheat your oven to 200°C (400°F).  
2. Clean the mushrooms and slice them.  
3. In a pan, heat some olive oil over medium heat and add the sliced mushrooms. Cook until they're golden brown. Season with salt and pepper.  
4. While the mushrooms are cooking, season the salmon fillet with salt and pepper.  
5. Place the salmon on a baking tray and add it to the oven. Bake for about 12-15 minutes or until the salmon is cooked through.  
6. Toast slices of whole wheat bread until golden.  
7. Once the salmon 

## 6.4 View full conversation history

In [23]:
if not conversation:
    print('No conversation yet. Use the chat cell above to start.')
else:
    print(f'Conversation history ({len(conversation)} messages):\n')
    for i, msg in enumerate(conversation, 1):
        role    = msg['role'].upper()
        content = str(msg['content'])[:200].replace('\n', ' ')
        print(f'[{i}] {role:<12} {content}...' if len(str(msg['content'])) > 200 else f'[{i}] {role:<12} {content}')

Conversation history (2 messages):

[1] USER         What should I cook tonight to use up what's about to expire?
[2] ASSISTANT    Here’s a great recipe to use up some of your at-risk ingredients!  **Recipe:** Salmon and Mushroom Toast   **Why this recipe?** Uses up salmon fillet (3 days), mushrooms (3 days), and whole wheat brea...


## 6.5 Reset conversation (start fresh)

In [24]:
conversation = []
print('Conversation cleared. Ready for a new session.')

Conversation cleared. Ready for a new session.
